# 3. Experiment Tracking with MLflow

This notebook demonstrates MLflow integration for experiment tracking:
- Loading pre-trained models from previous notebook
- MLflow setup and configuration
- Logging existing models with parameters, metrics, and artifacts
- Model comparison and registry
- Experiment visualization and analysis

## 3.1 Import Required Libraries

In [54]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML libraries
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
from xgboost import XGBClassifier

# MLflow for experiment tracking
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

# Utilities
import warnings
import os
import json
import joblib
from datetime import datetime
import time

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 3.2 Setup MLflow

In [55]:
# Setup MLflow tracking URI and experiment
mlflow_dir = "mlruns"
os.makedirs(mlflow_dir, exist_ok=True)

# Set tracking URI to local directory
mlflow.set_tracking_uri(f"file:///{os.path.abspath(mlflow_dir)}")

# Create or get experiment
experiment_name = "mlops_assignment_experiments"
try:
    experiment_id = mlflow.create_experiment(experiment_name)
    print(f"Created new experiment: {experiment_name}")
except:
    experiment = mlflow.get_experiment_by_name(experiment_name)
    experiment_id = experiment.experiment_id
    print(f"Using existing experiment: {experiment_name}")

mlflow.set_experiment(experiment_name)
print(f"Experiment ID: {experiment_id}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Using existing experiment: mlops_assignment_experiments
Experiment ID: 439973174090507163
Tracking URI: file:///c:\Users\avasin\Documents\MTech\Sem3\MLOps\Assign1\mlruns


## 3.3 Load Saved Models and Data

In [56]:
# Load saved model artifacts from previous notebook
print("Loading saved model artifacts...")

# Check if models directory and files exist
model_path = 'models/best_model_logistic_regression.pkl'


# Load best model
best_model = joblib.load(model_path)
print(f"Model loaded: {model_path}")
print(f"Model type: {type(best_model).__name__}")

# Load scaler
scaler = joblib.load('models/scaler.pkl')
print(f"Scaler loaded: models/scaler.pkl")

# Load feature names
with open('models/feature_names.json', 'r') as f:
    feature_info = json.load(f)
feature_names = feature_info['features']
print(f"Feature names loaded: {len(feature_names)} features")

# Load previous metrics
with open('models/model_metrics.json', 'r') as f:
    previous_metrics = json.load(f)
print(f"\nPrevious model performance:")
print(f"   Test Accuracy: {previous_metrics['test_accuracy']:.4f}")
print(f"   Test F1: {previous_metrics['test_f1']:.4f}")

# Load and prepare data
df = pd.read_csv('data/cleaned_data.csv')
print(f"\nData loaded: {df.shape}")

# Prepare features and target
X = df.drop('target', axis=1)
y = df['target']

# Encode categorical features if any
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
if len(categorical_cols) > 0:
    le = LabelEncoder()
    for col in categorical_cols:
        X[col] = le.fit_transform(X[col].astype(str))

# Split data (same split as before for consistency)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
X_train_scaled = pd.DataFrame(
    scaler.transform(X_train),
    columns=X.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

print(f"\nData prepared for experiment tracking:")
print(f"   Training set: {X_train_scaled.shape}")
print(f"   Test set: {X_test_scaled.shape}")

Loading saved model artifacts...
Model loaded: models/best_model_logistic_regression.pkl
Model type: LogisticRegression
Scaler loaded: models/scaler.pkl
Feature names loaded: 13 features

Previous model performance:
   Test Accuracy: 0.6066
   Test F1: 0.5839

Data loaded: (303, 14)

Data prepared for experiment tracking:
   Training set: (242, 13)
   Test set: (61, 13)


## 2.4 Training Function with MLflow Tracking

In [57]:
def log_model_to_mlflow(model, model_name, params, X_train, X_test, y_train, y_test, tags=None):
    """
    Log a pre-trained model to MLflow with all relevant information
    """
    with mlflow.start_run(run_name=model_name) as run:
        # Log tags
        if tags:
            mlflow.set_tags(tags)
        
        # Log parameters
        mlflow.log_params(params)
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("n_train_samples", X_train.shape[0])
        mlflow.log_param("n_test_samples", X_test.shape[0])
        mlflow.log_param("n_features", X_train.shape[1])
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate and log metrics
        train_accuracy = accuracy_score(y_train, y_train_pred)
        test_accuracy = accuracy_score(y_test, y_test_pred)
        test_precision = precision_score(y_test, y_test_pred, average='weighted')
        test_recall = recall_score(y_test, y_test_pred, average='weighted')
        test_f1 = f1_score(y_test, y_test_pred, average='weighted')
        
        mlflow.log_metric("train_accuracy", train_accuracy)
        mlflow.log_metric("test_accuracy", test_accuracy)
        mlflow.log_metric("test_precision", test_precision)
        mlflow.log_metric("test_recall", test_recall)
        mlflow.log_metric("test_f1_score", test_f1)
        mlflow.log_metric("overfit_gap", train_accuracy - test_accuracy)
        
        # Log confusion matrix as artifact
        cm = confusion_matrix(y_test, y_test_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix - {model_name}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        cm_path = f'confusion_matrix_{model_name}.png'
        plt.savefig(cm_path)
        plt.close()
        mlflow.log_artifact(cm_path)
        os.remove(cm_path)
        
        # Log classification report
        report = classification_report(y_test, y_test_pred, output_dict=True)
        report_path = f'classification_report_{model_name}.json'
        with open(report_path, 'w') as f:
            json.dump(report, f, indent=2)
        mlflow.log_artifact(report_path)
        os.remove(report_path)
        
        # Log model
        mlflow.sklearn.log_model(sk_model=model, artifact_path="model")
        
        # Log ROC AUC if binary classification
        if hasattr(model, 'predict_proba') and len(np.unique(y_test)) == 2:
            y_pred_proba = model.predict_proba(X_test)[:, 1]
            roc_auc = roc_auc_score(y_test, y_pred_proba)
            mlflow.log_metric("roc_auc_score", roc_auc)
            
            # Plot ROC curve
            fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
            plt.figure(figsize=(8, 6))
            plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
            plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            plt.title(f'ROC Curve - {model_name}')
            plt.legend(loc='lower right')
            plt.grid(alpha=0.3)
            roc_path = f'roc_curve_{model_name}.png'
            plt.savefig(roc_path)
            plt.close()
            mlflow.log_artifact(roc_path)
            os.remove(roc_path)
        
        run_id = run.info.run_id
        
        print(f"  {model_name} logged to MLflow")
        print(f"   Run ID: {run_id}")
        print(f"   Test Accuracy: {test_accuracy:.4f} | F1-Score: {test_f1:.4f}")
        
        return {
            'run_id': run_id,
            'model_name': model_name,
            'test_accuracy': test_accuracy,
            'test_f1': test_f1
        }

print("MLflow logging function defined!")

MLflow logging function defined!


## 3.5 Log Pre-trained Model to MLflow

In [58]:
print("=" * 70)
print("LOGGING PRE-TRAINED MODEL TO MLFLOW")
print("=" * 70)

# Extract model parameters
model_params = best_model.get_params()

# Create clean params dict for logging
log_params = {
    'C': model_params.get('C', 'N/A'),
    'penalty': model_params.get('penalty', 'N/A'),
    'solver': model_params.get('solver', 'N/A'),
    'max_iter': model_params.get('max_iter', 'N/A'),
    'random_state': model_params.get('random_state', 'N/A')
}

# Tags for the run
tags = {
    'experiment': 'production_model',
    'version': 'v2.0',
    'dataset': 'heart_disease',
    'source': 'notebook_02',
    'tuning_method': 'randomized_search'
}

# Log the model
result = log_model_to_mlflow(
    best_model,
    "Logistic_Regression_Production",
    log_params,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    tags
)

print("\n" + "=" * 70)
print("Model successfully logged to MLflow!")
print("=" * 70)

LOGGING PRE-TRAINED MODEL TO MLFLOW


2025/12/29 19:32:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Logistic_Regression_Production logged to MLflow
   Run ID: fb9918939cc64c4f8bd579d1663eefa7
   Test Accuracy: 0.6066 | F1-Score: 0.5839

Model successfully logged to MLflow!


## 3.6 Query MLflow Experiments

In [59]:
# Query runs from MLflow
client = MlflowClient()

# Get all runs from the experiment
experiment = mlflow.get_experiment_by_name(experiment_name)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

print(f"Total runs in experiment: {len(runs)}")
print("\nRun Details:")
display(runs[['run_id', 'start_time', 'tags.mlflow.runName', 
              'metrics.test_accuracy', 'metrics.test_f1_score']].head(10))

# Get the latest run (our logged model)
latest_run = runs.iloc[0]
best_run_id = latest_run['run_id']
print(f"\nLatest Run ID: {best_run_id}")
print(f"Model: {latest_run['tags.mlflow.runName']}")
print(f"Test Accuracy: {latest_run['metrics.test_accuracy']:.4f}")
print(f"F1-Score: {latest_run['metrics.test_f1_score']:.4f}")

Total runs in experiment: 6

Run Details:


,run_id,start_time,tags.mlflow.runName,metrics.test_accuracy,metrics.test_f1_score
0,fb9918939cc64c4f8bd579d1663eefa7,2025-12-29 14:02:16.441000+00:00,Logistic_Regression_Production,0.606557,0.583862
1,931b8d072b8343ddbc49e7cf91959564,2025-12-29 14:01:55.480000+00:00,Logistic_Regression_Production,0.606557,0.583862
2,aa786da06bd047fe9c857ca063f0734a,2025-12-29 14:01:11.788000+00:00,Logistic_Regression_Production,0.606557,0.583862
3,147cb22dfa0f428eabb16dd0b2ba1f0b,2025-12-29 13:55:49.793000+00:00,Logistic_Regression_Production,0.606557,0.583862
4,365baf60885d45cdac27a5b88ba65b1e,2025-12-29 13:55:28.792000+00:00,Logistic_Regression_Production,0.606557,0.583862
5,d6fb69e378bb47bdacdd00215e43c625,2025-12-29 13:50:19.475000+00:00,Logistic_Regression_Production,0.606557,0.583862



Latest Run ID: fb9918939cc64c4f8bd579d1663eefa7
Model: Logistic_Regression_Production
Test Accuracy: 0.6066
F1-Score: 0.5839


## 3.7 Model Registry

In [60]:
# Register model in MLflow registry
try:
    model_registry_name = "heart_disease_classifier"
    model_uri = f"runs:/{best_run_id}/model"
    model_details = mlflow.register_model(model_uri, model_registry_name)
    
    print(f"Model registered in MLflow:")
    print(f"   Model Name: {model_registry_name}")
    print(f"   Version: {model_details.version}")
    print(f"   Run ID: {best_run_id}")
    
    # Transition model to production
    client.transition_model_version_stage(
        name=model_registry_name,
        version=model_details.version,
        stage="Production"
    )
    print(f"   Stage: Production")
    
except Exception as e:
    print(f"Note: MLflow model registry not available in local tracking setup")
    print(f"Model artifacts available in MLflow tracking server")

Registered model 'heart_disease_classifier' already exists. Creating a new version of this model...
2025/12/29 19:32:21 WARNING mlflow.tracking._model_registry.fluent: Run with id fb9918939cc64c4f8bd579d1663eefa7 has no artifacts at artifact path 'model', registering model based on models:/m-fc48b769e86e4a4ca91ea6de91a97925 instead
Created version '5' of model 'heart_disease_classifier'.


Model registered in MLflow:
   Model Name: heart_disease_classifier
   Version: 5
   Run ID: fb9918939cc64c4f8bd579d1663eefa7
   Stage: Production


In [61]:
# Load and test model from MLflow
loaded_model_uri = f"runs:/{best_run_id}/model"
loaded_model = mlflow.sklearn.load_model(loaded_model_uri)

print(f"Model loaded from MLflow")
print(f"   Model type: {type(loaded_model).__name__}")

# Test loaded model
y_pred_loaded = loaded_model.predict(X_test_scaled)
accuracy_loaded = accuracy_score(y_test, y_pred_loaded)
f1_loaded = f1_score(y_test, y_pred_loaded, average='weighted')

print(f"\nLoaded model performance:")
print(f"   Test Accuracy: {accuracy_loaded:.4f}")
print(f"   F1-Score: {f1_loaded:.4f}")
print(f"\nModel successfully loaded and validated from MLflow!")

Model loaded from MLflow
   Model type: LogisticRegression

Loaded model performance:
   Test Accuracy: 0.6066
   F1-Score: 0.5839

Model successfully loaded and validated from MLflow!


## 2.8 Query MLflow Experiments

In [62]:
# Create reports directory
os.makedirs('reports', exist_ok=True)

# Save MLflow runs
mlflow_runs_csv = 'reports/mlflow_runs.csv'
runs.to_csv(mlflow_runs_csv, index=False)
print(f"MLflow runs saved: {mlflow_runs_csv}")

# Create experiment summary
summary = {
    'experiment_name': experiment_name,
    'experiment_id': experiment_id,
    'total_runs': len(runs),
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'production_model': {
        'name': 'Logistic_Regression_Production',
        'run_id': best_run_id,
        'test_accuracy': float(latest_run['metrics.test_accuracy']),
        'test_f1': float(latest_run['metrics.test_f1_score']),
        'source': 'notebook_02_saved_model'
    },
    'dataset_info': {
        'n_samples': int(len(df)),
        'n_features': int(X.shape[1]),
        'n_train': int(X_train.shape[0]),
        'n_test': int(X_test.shape[0])
    }
}

summary_file = 'reports/mlflow_experiment_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Experiment summary saved: {summary_file}")

MLflow runs saved: reports/mlflow_runs.csv
Experiment summary saved: reports/mlflow_experiment_summary.json


## 2.10 Load and Test Logged Model

In [64]:
print("=" * 80)
print("MLFLOW EXPERIMENT TRACKING SUMMARY")
print("=" * 80)

print(f"\nSOURCE:")
print(f"   - Loaded pre-trained model from: {model_path}")
print(f"   - Model trained in notebook: 02_feature_engineering_model_development.ipynb")

print(f"\nDATA:")
print(f"   - Features: {X.shape[1]}")
print(f"   - Training samples: {X_train_scaled.shape[0]:,}")
print(f"   - Test samples: {X_test_scaled.shape[0]:,}")

print(f"\nMLFLOW EXPERIMENT:")
print(f"   - Experiment Name: {experiment_name}")
print(f"   - Experiment ID: {experiment_id}")
print(f"   - Total Runs: {len(runs)}")
print(f"   - Tracking URI: {mlflow.get_tracking_uri()}")

print(f"\nLOGGED MODEL:")
print(f"   - Model Type: {type(best_model).__name__}")
print(f"   - Test Accuracy: {latest_run['metrics.test_accuracy']:.4f}")
print(f"   - F1-Score: {latest_run['metrics.test_f1_score']:.4f}")
print(f"   - Run ID: {best_run_id}")

print(f"\nSAVED ARTIFACTS:")
print(f"   - MLflow Runs: {mlflow_runs_csv}")
print(f"   - Summary: {summary_file}")
print(f"   - MLflow Directory: {mlflow_dir}")

print(f"\nKEY FEATURES:")
print(f"   - Model parameters logged to MLflow")
print(f"   - Performance metrics tracked (accuracy, precision, recall, F1, ROC-AUC)")
print(f"   - Artifacts saved (confusion matrix, ROC curve, classification report)")
print(f"   - Model registered for deployment")
print(f"   - Experiments queryable via MLflow UI")

print("\n" + "=" * 80)
print("MLflow Experiment Tracking completed!")
print("=" * 80)

MLFLOW EXPERIMENT TRACKING SUMMARY

SOURCE:
   - Loaded pre-trained model from: models/best_model_logistic_regression.pkl
   - Model trained in notebook: 02_feature_engineering_model_development.ipynb

DATA:
   - Features: 13
   - Training samples: 242
   - Test samples: 61

MLFLOW EXPERIMENT:
   - Experiment Name: mlops_assignment_experiments
   - Experiment ID: 439973174090507163
   - Total Runs: 6
   - Tracking URI: file:///c:\Users\avasin\Documents\MTech\Sem3\MLOps\Assign1\mlruns

LOGGED MODEL:
   - Model Type: LogisticRegression
   - Test Accuracy: 0.6066
   - F1-Score: 0.5839
   - Run ID: fb9918939cc64c4f8bd579d1663eefa7

SAVED ARTIFACTS:
   - MLflow Runs: reports/mlflow_runs.csv
   - Summary: reports/mlflow_experiment_summary.json
   - MLflow Directory: mlruns

KEY FEATURES:
   - Model parameters logged to MLflow
   - Performance metrics tracked (accuracy, precision, recall, F1, ROC-AUC)
   - Artifacts saved (confusion matrix, ROC curve, classification report)
   - Model registe